# 🏥 3D DICOM Fine-Tuning for Multi-Label Pathology Classification

**Workflow:** Load preprocessed 3D volumes (.npy) → Fine-tune 3D-ResNet50 → Evaluate per-class metrics

**Prerequisites:** Run `Preprocess_DICOM_to_Volumes_3D.ipynb` first to generate volumes

## 📦 Section 1: Install Dependencies and Imports

In [ ]:
import subprocess
import sys

# Install required packages
packages = ['torch', 'torchvision', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models

import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import json
from datetime import datetime

print("✅ All packages imported successfully!")
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

## 🔧 Section 2: Mount Google Drive & Configure Paths

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = '/content/drive/MyDrive/artishow'
    print("✅ Google Drive mounted")
except ImportError:
    # Local development
    DRIVE_FOLDER = '/home/infres/ahmed-25/artishow'
    print("⚠️ Running locally (no Google Drive)")

# Configure paths
VOLUMES_CSV = os.path.join(DRIVE_FOLDER, 'dataset_labeled_volumes_3d.csv')
OUTPUT_FOLDER = os.path.join(DRIVE_FOLDER, 'dicom_volume', 'model_outputs_3d')

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"\n📁 Paths:")
print(f"  Volumes CSV: {VOLUMES_CSV}")
print(f"  Output folder: {OUTPUT_FOLDER}")

# Verify CSV exists
if os.path.exists(VOLUMES_CSV):
    print(f"✅ Volumes CSV found!")
else:
    print(f"❌ ERROR: Volumes CSV not found!")
    print(f"   Run Preprocess_DICOM_to_Volumes_3D.ipynb first")

## 📊 Section 3: Load Preprocessed Volumes Index & Labels

In [ ]:
# Load preprocessed volumes index
df_labels = pd.read_csv(VOLUMES_CSV)

print(f"{'='*70}")
print("📊 PREPROCESSED VOLUMES DATASET")
print(f"{'='*70}")
print(f"Total volumes: {len(df_labels)}")
print(f"Shape: {df_labels.shape}")
print(f"\nFirst 3 rows:")
print(df_labels[['image_id', 'volume_npy_path', 'Normal']].head(3))

# Verify volumes exist
missing = sum(1 for path in df_labels['volume_npy_path'] if not os.path.exists(path))
if missing > 0:
    print(f"\n⚠️ {missing} volumes missing!")
    df_labels = df_labels[df_labels['volume_npy_path'].apply(lambda x: os.path.exists(x))]
else:
    print(f"\n✅ All {len(df_labels)} volume files verified!")

# Define pathology classes
PATHOLOGY_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Pneumonia', 'Pneumothorax',
    'Edema', 'Emphysema', 'Fibrosis', 'Infiltration', 'Mass', 'Nodule',
    'Hernia', 'Fracture', 'Pleural_Thickening', 'Opacity', 'Consolidation',
    'Granuloma', 'Calcinosis', 'Scoliosis', 'Atherosclerosis', 'Normal'
]

print(f"\n🏷️ Pathology classes: {len(PATHOLOGY_CLASSES)}")
label_counts = df_labels[PATHOLOGY_CLASSES].sum().sort_values(ascending=False)
for label, count in label_counts.head(5).items():
    pct = count / len(df_labels) * 100
    print(f"  {label:20s}: {int(count):4d} ({pct:5.1f}%)")

## 📦 Section 4: Helper Functions for Volume Loading

In [ ]:
def load_volume_npy(volume_path):
    """
    Load preprocessed volume from .npy file.
    Volumes are already windowed, resized, and normalized!
    """
    try:
        volume = np.load(volume_path)
        return volume
    except Exception as e:
        print(f"Error loading {volume_path}: {e}")
        return None

print("✅ Volume loading function defined")
print("📌 Note: Volumes are ALREADY preprocessed (windowed, resized, normalized)")

## 🔀 Section 5: Create Train/Val/Test Splits

In [ ]:
# Create splits (70/15/15)
train_df, temp_df = train_test_split(df_labels, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"{'='*70}")
print("🔀 DATASET SPLIT (70/15/15)")
print(f"{'='*70}")
print(f"Train: {len(train_df):4d} ({len(train_df)/len(df_labels)*100:5.1f}%)")
print(f"Val:   {len(val_df):4d} ({len(val_df)/len(df_labels)*100:5.1f}%)")
print(f"Test:  {len(test_df):4d} ({len(test_df)/len(df_labels)*100:5.1f}%)")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)}")

# Save splits
train_df.to_csv(os.path.join(OUTPUT_FOLDER, 'split_train_3d.csv'), index=False)
val_df.to_csv(os.path.join(OUTPUT_FOLDER, 'split_val_3d.csv'), index=False)
test_df.to_csv(os.path.join(OUTPUT_FOLDER, 'split_test_3d.csv'), index=False)
print(f"\n✅ Splits saved")

## 🎨 Section 6: Define 3D Augmentations

In [ ]:
class Random3DFlip(nn.Module):
    """Apply random 3D flips"""
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
    
    def forward(self, volume):
        if np.random.random() < self.p:
            volume = torch.flip(volume, [2])  # Flip height
        if np.random.random() < self.p:
            volume = torch.flip(volume, [3])  # Flip width
        return volume

# Training transforms (with augmentation)
train_transforms = Random3DFlip(p=0.3)

# Validation/Test transforms (no augmentation)
val_transforms = nn.Identity()

print("✅ 3D augmentations defined")
print("  Train: Random flips (30% probability)")
print("  Val/Test: No augmentation")

## 🏗️ Section 7: Define 3D PyTorch Dataset Class

In [ ]:
class DICOM3DMultiLabelDataset(Dataset):
    """Multi-label 3D Dataset loading preprocessed .npy volumes"""
    
    def __init__(self, dataframe, label_columns, transform=None):
        """
        Args:
            dataframe: DataFrame with 'volume_npy_path' + label columns
            label_columns: List of 21 pathology column names
            transform: Augmentation pipeline
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.label_columns = label_columns
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        volume_path = row['volume_npy_path']
        image_id = row['image_id']
        
        # Load preprocessed volume from .npy
        volume = load_volume_npy(volume_path)
        
        if volume is None:
            volume = np.zeros((128, 128, 64), dtype=np.float32)
        
        # Convert to tensor and add channel dimension [1, H, W, D]
        volume_tensor = torch.from_numpy(volume).float().unsqueeze(0)
        
        # Apply transforms (augmentation)
        if self.transform:
            volume_tensor = self.transform(volume_tensor)
        
        # Get labels (21 binary values)
        labels = torch.tensor(row[self.label_columns].values.astype(np.float32))
        
        return volume_tensor, labels, image_id

print("✅ 3D Dataset class defined")
print(f"   Input: .npy volumes (128×128×64, already normalized)")
print(f"   Output: [1, 128, 128, 64] tensor + [21] labels")

## 📊 Section 8: Create DataLoaders

In [ ]:
BATCH_SIZE = 4
NUM_WORKERS = 2

print(f"\n{'='*70}")
print("📊 Creating 3D DICOM DataLoaders")
print(f"{'='*70}")

# Create datasets
train_dataset = DICOM3DMultiLabelDataset(
    train_df, PATHOLOGY_CLASSES, transform=train_transforms
)
val_dataset = DICOM3DMultiLabelDataset(
    val_df, PATHOLOGY_CLASSES, transform=val_transforms
)
test_dataset = DICOM3DMultiLabelDataset(
    test_df, PATHOLOGY_CLASSES, transform=val_transforms
)

print(f"Train dataset: {len(train_dataset)}")
print(f"Val dataset:   {len(val_dataset)}")
print(f"Test dataset:  {len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\n✅ DataLoaders created")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

## 🤖 Section 9: Load 3D-ResNet50 Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load 3D-ResNet18 (Kinetics pretrained)
print("\n📥 Loading 3D-ResNet18 (Kinetics pretrained)...")
try:
    model = models.video.r3d_18(pretrained=True)
    print("✅ Successfully loaded pretrained model")
except:
    print("⚠️ Using non-pretrained model")
    model = models.video.r3d_18(pretrained=False)

# Freeze early layers
for param in model.layer1.parameters():
    param.requires_grad = False
for param in model.layer2.parameters():
    param.requires_grad = False

# Replace classifier for MULTI-LABEL (21 outputs)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, len(PATHOLOGY_CLASSES))  # 21 outputs
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model configured for MULTI-LABEL classification")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Output shape: [batch_size, {len(PATHOLOGY_CLASSES)}]")

## 🏋️ Section 10: Training Configuration

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-4
PATIENCE = 3

# Multi-label loss
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=1e-5
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_counter = 0

print(f"{'='*70}")
print("🏋️ TRAINING CONFIGURATION")
print(f"{'='*70}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Loss function: BCEWithLogitsLoss (Multi-label)")
print(f"Device: {device}")

## 🔄 Section 11: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train one epoch"""
    model.train()
    total_loss = 0
    
    for volumes, labels, _ in tqdm(loader, desc="Training"):
        volumes = volumes.to(device)
        labels = labels.to(device)
        
        # Forward
        outputs = model(volumes)
        loss = criterion(outputs, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    """Validate one epoch"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for volumes, labels, _ in tqdm(loader, desc="Validating"):
            volumes = volumes.to(device)
            labels = labels.to(device)
            
            outputs = model(volumes)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    
    return total_loss / len(loader)

print("✅ Training functions defined")

In [ ]:
# Main training loop
print(f"\n{'='*70}")
print("🚀 STARTING TRAINING")
print(f"{'='*70}\n")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    history['train_loss'].append(train_loss)
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    history['val_loss'].append(val_loss)
    
    # Scheduler
    scheduler.step(val_loss)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_FOLDER, 'best_model_3d.pth'))
        print("✅ Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹️ Early stopping")
            break

print(f"\n{'='*70}")
print("✅ TRAINING COMPLETE")
print(f"{'='*70}")

## 📊 Section 12: Training History Visualization

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
ax.plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('3D Model Training History', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'training_history_3d.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Training history plot saved")

## 🧪 Section 13: Test Set Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load(os.path.join(OUTPUT_FOLDER, 'best_model_3d.pth')))

# Test evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []

print("\n🧪 Evaluating on test set...")
with torch.no_grad():
    for volumes, labels, _ in tqdm(test_loader):
        volumes = volumes.to(device)
        outputs = model(volumes)
        
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

# Convert to arrays
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Calculate metrics
test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
test_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"\n{'='*70}")
print("📊 TEST SET RESULTS")
print(f"{'='*70}")
print(f"Accuracy:  {test_acc*100:.2f}%")
print(f"Precision: {test_precision*100:.2f}%")
print(f"Recall:    {test_recall*100:.2f}%")
print(f"F1-Score:  {test_f1*100:.2f}%")

## 💾 Section 14: Save Results

In [ ]:
# Save final model
torch.save(model.state_dict(), os.path.join(OUTPUT_FOLDER, 'model_3d_final.pth'))

# Save metrics
results = {
    'model': '3D-ResNet18',
    'epochs_trained': len(history['train_loss']),
    'test_accuracy': float(test_acc),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'test_f1': float(test_f1),
    'timestamp': datetime.now().isoformat(),
    'training_history': {
        'train_loss': [float(x) for x in history['train_loss']],
        'val_loss': [float(x) for x in history['val_loss']]
    }
}

with open(os.path.join(OUTPUT_FOLDER, 'results_3d.json'), 'w') as f:
    json.dump(results, f, indent=4)

print(f"\n{'='*70}")
print("💾 FILES SAVED")
print(f"{'='*70}")
print(f"Model: {os.path.join(OUTPUT_FOLDER, 'model_3d_final.pth')}")
print(f"Results: {os.path.join(OUTPUT_FOLDER, 'results_3d.json')}")
print(f"Plot: {os.path.join(OUTPUT_FOLDER, 'training_history_3d.png')}")